### To download the report about gilts, go here: https://www.dmo.gov.uk/data/pdfdatareport?reportCode=D1A

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import pandas as pd
from datetime import datetime

### Assumptions:
- Assumes 20% tax on pension (lowest tax rate in UK, ~lowest tax rates + skladka zdrowotna in Poland)
- Assumes no changes to the taxes
- Assumes the annual return rates for the last 10 years for both SP500 and the pension fund I am in
- Assumes capital gains tax allowance is used to the fullest each year (3,000 a year) and doesn't change over time

### Simplifications
- Ignores the 30 day cooling period when selling SP500 to use the capital gains tax allowance
- Ignores the fact that pension can only be acessed at 57 while ETF money can be accessed at any time
- Ignores taxes on dividends from ETFs (but there is an allowance of 500 a year)
- Ignores a 25% free of tax lump sum payment for the pension (probably not valid when receiving pension abroad)


In [67]:
# Parameters
years = 10
pension_return = 0.0959
etf_return = 0.1539
pension_tax = 0.20
cg_tax = 0.24
pre_tax_etf = 0.45  # 45% upfront tax on ETF contribution

# Annual contributions
split_pension_contrib = 20400
split_etf_contrib = 39600
all_pension_contrib = 60000

# --- Split strategy: start-of-year ---
def simulate_split_start():
    pension_balance = 0
    etf_balance = 0
    etf_cost_basis = 0
    for year in range(years):
        # Contributions at start
        pension_balance += split_pension_contrib
        etf_net_contrib = split_etf_contrib * (1 - pre_tax_etf)
        etf_balance += etf_net_contrib
        etf_cost_basis += etf_net_contrib

        # Apply annual returns
        pension_balance *= (1 + pension_return)
        etf_balance *= (1 + etf_return)

    # Taxes
    pension_balance *= (1 - pension_tax)

    # ETF gains taxed at end
    etf_gain = etf_balance - etf_cost_basis
    tax = etf_gain * cg_tax if etf_gain > 0 else 0

    # Apply allowance: 10 years × 3000 = 30,000
    tax -= 3000 * 10
    tax = max(tax, 0)  # tax cannot be negative
    etf_balance -= tax

    return pension_balance + etf_balance

# --- Split strategy: end-of-year ---
def simulate_split_end():
    pension_balance = 0
    etf_balance = 0
    etf_cost_basis = 0
    for year in range(years):
        # Apply returns first
        pension_balance *= (1 + pension_return)
        etf_balance *= (1 + etf_return)

        # Add contributions at end
        pension_balance += split_pension_contrib
        etf_net_contrib = split_etf_contrib * (1 - pre_tax_etf)
        etf_balance += etf_net_contrib
        etf_cost_basis += etf_net_contrib

    # Taxes
    pension_balance *= (1 - pension_tax)

    # ETF gains taxed at end
    etf_gain = etf_balance - etf_cost_basis
    tax = etf_gain * cg_tax if etf_gain > 0 else 0

    # Apply allowance: 9 years × 3000 = 27,000
    tax -= 3000 * 9
    tax = max(tax, 0)
    etf_balance -= tax

    return pension_balance + etf_balance

# --- All-pension strategy: start-of-year ---
def simulate_all_start():
    pension_balance = 0
    for year in range(years):
        pension_balance += all_pension_contrib
        pension_balance *= (1 + pension_return)
    pension_balance *= (1 - pension_tax)
    return pension_balance

# --- All-pension strategy: end-of-year ---
def simulate_all_end():
    pension_balance = 0
    for year in range(years):
        pension_balance *= (1 + pension_return)
        pension_balance += all_pension_contrib
    pension_balance *= (1 - pension_tax)
    return pension_balance

# Run simulations
results = {
    "All-Pension (year-start)": simulate_all_start(),
    "JS Match (year-start)": simulate_split_start(),
    "All-Pension (year-end)": simulate_all_end(),
    "JS Match (year-end)": simulate_split_end()
}

# Display results
for strategy, total in results.items():
    print(f"{strategy}: £{total:,.0f}")

# Calculate ratios
ratio_start = results["All-Pension (year-start)"] / results["JS Match (year-start)"]
ratio_end = results["All-Pension (year-end)"] / results["JS Match (year-end)"]

print(f"Pension / ETF ratio (year-start): {ratio_start:.4f} ({ratio_start*100:.2f}%)")
print(f"Pension / ETF ratio (year-end): {ratio_end:.4f} ({ratio_end*100:.2f}%)")

All-Pension (year-start): £822,054
JS Match (year-start): £757,039
All-Pension (year-end): £750,117
JS Match (year-end): £676,862
Pension / ETF ratio (year-start): 1.0859 (108.59%)
Pension / ETF ratio (year-end): 1.1082 (110.82%)


### Load current gilts prices

In [ ]:
# -----------------------------
# Step 1: Load the web page
# -----------------------------
url = "https://www.hl.co.uk/shares/corporate-bonds-gilts/bond-prices/uk-gilts?column=maturity&order=asc"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
response = requests.get(url, headers=headers)

if response.status_code != 200:
    raise Exception(f"Failed to load page, status code: {response.status_code}")

html = response.text

# -----------------------------
# Step 2: Parse with BeautifulSoup
# -----------------------------
soup = BeautifulSoup(html, "html.parser")

# Find the table containing gilts
# The page uses <table> for the bond prices
table = soup.find("table")  # assuming first table is the correct one

# -----------------------------
# Step 3: Extract table rows
# -----------------------------
rows = table.find_all("tr")[1:]  # skip header row

data = []

for row in rows:
    cells = row.find_all("td")
    if len(cells) < 5:
        continue  # skip incomplete rows

    # --- Issuer ---
    issuer_tag = cells[0].find("a", class_="link-headline")
    if issuer_tag:
        issuer_text = issuer_tag.get_text(strip=True)
    else:
        issuer_text = ""

    # Extract coupon from issuer text (e.g., "Treasury 1.5% 22/07/2026")
    coupon_match = re.search(r"(\d+\.?\d*)%", issuer_text)
    coupon = float(coupon_match.group(1)) if coupon_match else None

    # Extract maturity date from issuer text
    maturity_match = re.search(r"(\d{2}/\d{2}/\d{4})", issuer_text)
    maturity = maturity_match.group(1) if maturity_match else ""

    # --- Price ---
    price_text = cells[2].get_text(strip=True).replace(",", "")
    try:
        price = float(price_text)
    except:
        price = None

    # Actions column (usually links/buttons)
    actions = cells[4].get_text(strip=True)

    data.append({
        "Issuer": issuer_text,
        "Coupon (%)": coupon,
        "Maturity": maturity,
        "Price": price,
        "Actions": actions
    })

# -----------------------------
# Step 4: Create DataFrame
# -----------------------------
gilts_df = pd.DataFrame(data)
print(gilts_df.head())

### Define helper functions

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------

def parse_date(date_str):
    """Parse a date string like '31 Jan 2028'"""
    return datetime.strptime(date_str.strip(), "%d %b %Y")

def accrued_interest(clean_price, coupon_rate, settlement_date, last_coupon_date, face_value=100):
    """
    Calculate accrued interest to add to clean price.
    """
    days_between_coupons = (settlement_date - last_coupon_date).days
    coupon_period_days = 182.5  # approx 6 months
    semi_coupon = coupon_rate / 2 * face_value
    return semi_coupon * (days_between_coupons / coupon_period_days)

def generate_coupon_dates(first_issue, redemption, dividend_dates):
    """
    Generate all coupon dates between first issue and redemption.
    dividend_dates: string like '22 Jan/Jul'
    """
    start_year = first_issue.year
    end_year = redemption.year
    schedule = []

    day_months = []
    for dm in dividend_dates.split('/'):
        day, mon = dm.split()
        day_months.append((int(day), mon))

    for y in range(start_year, end_year + 1):
        for day, mon in day_months:
            try:
                dt = datetime.strptime(f"{day} {mon} {y}", "%d %b %Y")
            except:
                continue
            if dt > first_issue and dt <= redemption:
                schedule.append(dt)
    return sorted(schedule)

def simplified_yield(dirty_price, coupons, face_value, years_to_maturity):
    """
    Compute after-tax annual yield assuming all cashflows are at maturity
    """
    total_inflows = sum(coupons) + face_value
    r = (total_inflows / dirty_price) ** (1 / years_to_maturity) - 1
    return r

### Example usage

In [ ]:
# -----------------------------
# Example input tables
# -----------------------------

# Market table (clean prices)
market_data = pd.DataFrame({
    "Issuer": ["Treasury 0.125% 31/01/2028"],
    "Coupon (%)": [0.125],
    "Maturity": ["31 Jan 2028"],
    "Price": [92.490]  # clean price
})

# Gilt details table
details_data = pd.DataFrame({
    "Conventional Gilts": ["Treasury 0.125% 31/01/2028"],
    "ISIN Code": ["GB00BMBL1G81"],
    "Redemption Date": ["31 Jan 2028"],
    "First Issue Date": ["22 Jan 2023"],
    "Dividend Dates": ["22 Jan/Jul"],
    "Ex-dividend Date": ["22 Jan 2028"],
    "Amount in Issue (£ million nominal)": ["1000"]
})

# Settlement date
settlement = datetime(2025, 8, 31)

# -----------------------------
# Compute simplified after-tax yield for each gilt
# -----------------------------

results = []

for _, row in market_data.iterrows():
    clean_price = row["Price"]
    coupon_rate = row["Coupon (%)"]
    maturity = parse_date(row["Maturity"])

    # Match gilt details
    details = details_data[details_data["Conventional Gilts"] == row["Issuer"]].iloc[0]
    first_issue = parse_date(details["First Issue Date"])
    dividend_dates = details["Dividend Dates"]

    # Coupon schedule
    coupon_schedule = generate_coupon_dates(first_issue, maturity, dividend_dates)
    last_coupon = max([d for d in coupon_schedule if d <= settlement])

    # Dirty price = clean + accrued interest
    dirty_price = clean_price + accrued_interest(clean_price, coupon_rate, settlement, last_coupon)

    # Number of semi-annual coupons remaining (after settlement)
    remaining_coupons = [c for c in coupon_schedule if c > settlement]
    semi_coupon_value = coupon_rate / 2 * 100  # per £100 face
    coupons_after_tax = [semi_coupon_value * (1 - 0.45) for _ in remaining_coupons]

    # Fractional years to maturity
    years_to_maturity = (maturity - settlement).days / 365

    # Simplified annual yield
    r = simplified_yield(dirty_price, coupons_after_tax, 100, years_to_maturity)

    results.append({
        "Gilt": row["Issuer"],
        "Dirty Price": round(dirty_price, 3),
        "After-tax annual yield": round(r * 100, 4)
    })

# -----------------------------
# Show results
# -----------------------------
results_df = pd.DataFrame(results)
print(results_df)